In [3]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Select metric type here ("std" or "ci")
STAT_TYPE = "std"  # Change to "ci" whenever you want Confidence Intervals


def summarize_metric(group, column, stat_type=STAT_TYPE, confidence=0.95):
    """Summarizes a column as Mean ± [Std | CI] (raw_values)."""
    values = group[column].dropna().to_numpy(dtype=float)
    n = len(values)

    if n == 0:
        return "N/A"

    mean = np.mean(values)

    if n > 1:
        if stat_type == "std":
            margin = np.std(values, ddof=1)
        elif stat_type == "ci":
            sem = stats.sem(values)
            margin = sem * stats.t.ppf((1 + confidence) / 2, df=n - 1)
        else:
            raise ValueError("stat_type must be either 'std' or 'ci'")

        margin_str = f"{margin:.2f}"
    else:
        margin_str = "N/A"

    actual = ", ".join(f"{x:.2f}" for x in values)

    return f"{mean:.2f} ± {margin_str} ({actual})"




from pathlib import Path
import json

import numpy as np
import pandas as pd


def summarize_run(run_dir, tail_n=5):
    run_dir = Path(run_dir)

    with open(run_dir / "args.json", encoding="utf-8") as f:
        args = json.load(f)

    fid_path = run_dir / "validation_fid.jsonl"
    fids = []

    if fid_path.exists():
        with open(fid_path, encoding="utf-8") as f:
            fids = [
                json.loads(line)
                for line in f
                if line.strip()
            ]

    row = {
        "run": run_dir.name,
        **{f"arg_{key}": value for key, value in args.items()},
        "fid_best": np.nan,
        "fid_best_iteration": np.nan,
        "fid_final": np.nan,
        "fid_final_iteration": np.nan,
        "fid_tail_median": np.nan,
        "fid_tail_mean": np.nan,
        "fid_tail_max": np.nan,
        "fid_tail_n": 0,
    }

    if fids:
        values = np.array([item["fid"] for item in fids], dtype=float)
        iterations = np.array([item["iteration"] for item in fids])

        tail = values[-tail_n:]

        best_index = np.argmin(values)

        row.update({
            "fid_best": values[best_index],
            "fid_best_iteration": iterations[best_index],
            "fid_final": values[-1],
            "fid_final_iteration": iterations[-1],
            "fid_tail_median": np.median(tail),
            "fid_tail_mean": np.mean(tail),
            "fid_tail_max": np.max(tail),
            "fid_tail_n": len(tail),
        })

    return row

def summarize_runs(experiment_dirs, tail_n=5):
    """Summarizes all run subdirectories found inside multiple experiment directories."""
    rows = []

    # Handle both a single path string/Path object and a list of paths
    if isinstance(experiment_dirs, (str, Path)):
        experiment_dirs = [experiment_dirs]

    for exp_dir in experiment_dirs:
        exp_path = Path(exp_dir)

        # Iterate through subdirectories inside each experiment root
        for run_dir in exp_path.iterdir():
            # Check if it's a directory and contains args.json
            if run_dir.is_dir() and (run_dir / "args.json").exists():
                rows.append(summarize_run(run_dir, tail_n=tail_n))

    return pd.DataFrame(rows)

# Example usage:


In [4]:
# 2. Build comparison DataFrame
suffix = "std" if STAT_TYPE == "std" else "95ci"

dirs = [
    "/home/satoshi/projects/fcmstylegan/experiments/eeeg/diffusion/",
]
df = summarize_runs(dirs)


comparison = (
    df.groupby(
        [   
            "arg_seed",
            "arg_backbone",
            "arg_objective",
            "arg_sampler",
            "arg_profile_encoder",
            "fid_final_iteration",
            "arg_profile_encoder",
            
        ],
        dropna=False,
    )
    .apply(
        lambda g: pd.Series({
            "runs": len(g),
            f"tail_fid_{suffix}": summarize_metric(g, "fid_tail_median", stat_type=STAT_TYPE),
            f"best_fid_{suffix}": summarize_metric(g, "fid_best", stat_type=STAT_TYPE),
            f"final_fid_{suffix}": summarize_metric(g, "fid_final", stat_type=STAT_TYPE),
            f"tail_max_fid_{suffix}": summarize_metric(g, "fid_tail_max", stat_type=STAT_TYPE),
        })
    )
)

display(comparison)

runs  \
arg_seed arg_backbone arg_objective arg_sampler arg_profile_encoder fid_final_iteration arg_profile_encoder         
123      adm          ddpm          auto        cnn                 300000              cnn                     1   
                      edm           auto        cnn                 275000              cnn                     1   
                                                mlp                 295000              mlp                     1   
         compact      ddpm          auto        cnn                 300000              cnn                     1   
                                                mlp                 140000              mlp                     1   
                                    ddpm        cnn                 195000              cnn                     1   
                                                mlp                 60000               mlp                     1   

                                                                                                                    tail_fid_std  \
arg_seed arg_backbone arg_objective arg_sampler arg_profile_encoder fid_final_iteration arg_profile_encoder                        
123      adm          ddpm          auto        cnn                 300000              cnn                  20.04 ± N/A (20.04)   
                      edm           auto        cnn                 275000              cnn                  17.53 ± N/A (17.53)   
                                                mlp                 295000              mlp                  15.09 ± N/A (15.09)   
         compact      ddpm          auto        cnn                 300000              cnn                  18.50 ± N/A (18.50)   
                                                mlp                 140000              mlp                  30.30 ± N/A (30.30)   
                                    ddpm        cnn                 195000              cnn                  13.51 ± N/A (13.51)   
                                                mlp                 60000               mlp                  23.10 ± N/A (23.10)   

                                                                                                                    best_fid_std  \
arg_seed arg_backbone arg_objective arg_sampler arg_profile_encoder fid_final_iteration arg_profile_encoder                        
123      adm          ddpm          auto        cnn                 300000              cnn                  19.33 ± N/A (19.33)   
                      edm           auto        cnn                 275000              cnn                  11.88 ± N/A (11.88)   
                                                mlp                 295000              mlp                  11.92 ± N/A (11.92)   
         compact      ddpm          auto        cnn                 300000              cnn                  18.11 ± N/A (18.11)   
                                                mlp                 140000              mlp                  28.75 ± N/A (28.75)   
                                    ddpm        cnn                 195000              cnn                  12.46 ± N/A (12.46)   
                                                mlp                 60000               mlp                  18.79 ± N/A (18.79)   

                                                                                                                   final_fid_std  \
arg_seed arg_backbone arg_objective arg_sampler arg_profile_encoder fid_final_iteration arg_profile_encoder                        
123      adm          ddpm          auto        cnn                 300000              cnn                  19.33 ± N/A (19.33)   
                      edm           auto        cnn                 275000              cnn                  16.12 ± N/A (16.12)   
                                                mlp                 295000              mlp                  15.28 ± N/A (15.28)   
         compact 